In [1]:
from mikeio import Mesh, Dfsu
from datetime import datetime,timedelta
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('png')
plt.rcParams["figure.figsize"] = (10,8)
import pyevtk

In [2]:
dfsuFileName="area.dfsu"
dfsu = Dfsu(dfsuFileName)

In [3]:
Dfsu.show_progress = True # Turn on progress bar, useful to get progress for long-running tasks
ds = dfsu.read()

100%|█████████████████████████████████████████████████████████████████████████████| 2161/2161 [00:02<00:00, 849.11it/s]


In [4]:
msh = Mesh(dfsuFileName)
msh

Number of elements: 12321
Number of nodes: 8880
Projection: PROJCS["WGS_1984_UTM_Zone_30N",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["Degree",0.017453292519943295]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000],PARAMETER["False_Northing",0],PARAMETER["Central_Meridian",-3],PARAMETER["Scale_Factor",0.9996],PARAMETER["Latitude_Of_Origin",0],UNIT["Meter",1]]

In [5]:
msh.is_2d

True

In [6]:
x=np.ascontiguousarray(msh.node_coordinates[:,0])
y=np.ascontiguousarray(msh.node_coordinates[:,1])
z=np.ascontiguousarray(msh.node_coordinates[:,2])

offsets=[]
i=0
for j in range(msh.n_elements):
    m=len(msh.element_table[j])
    i+=m
    offsets.append(i)
offsets=np.array(offsets)

cellTypePoly=np.empty(msh.n_elements)
cellTypePoly.fill(7)

In [7]:
pvdObj=pyevtk.vtk.VtkGroup(dfsuFileName[0:-5])
time=dfsu.start_time
timeStep=timedelta(seconds=dfsu.timestep)
countTimeStep=0

while time <= dfsu.end_time:
    #print(str(time), countTimeStep)
    cellData={}
    for item in dfsu.items:
        timeString=str(time).replace(" ", "_")
        timeString=str(time).replace(":", "-")
        cellData[item.name]=ds[[item.name]][0][countTimeStep]
    pyevtk.hl.unstructuredGridToVTK(dfsuFileName[0:-5]+'_'+timeString, x=x, y=y, z=z, connectivity=np.concatenate(msh.element_table,axis=None), offsets=offsets, cell_types=cellTypePoly, cellData = cellData)
    pvdObj.addFile(dfsuFileName[0:-5]+'_'+timeString+'.vtu', countTimeStep)
    time+=timeStep
    countTimeStep+=1

pvdObj.save()

C:\Users\hels\Anaconda3\envs\feb22\lib\site-packages\mikeio\dataset.py:258: UserWarning: Indexing in MIKE IO 1.0 will not return a numpy array, but a mikeio.DataArray. More info: https://github.com/DHI/mikeio#readme
  warnings.warn(


In [4]:
dfsu

Dfsu2D
Number of elements: 12321
Number of nodes: 8880
Projection: PROJCS["WGS_1984_UTM_Zone_30N",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["Degree",0.017453292519943295]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000],PARAMETER["False_Northing",0],PARAMETER["Central_Meridian",-3],PARAMETER["Scale_Factor",0.9996],PARAMETER["Latitude_Of_Origin",0],UNIT["Meter",1]]
Items:
  0:  Surface elevation <Surface Elevation> (meter)
  1:  Total water depth <Water Depth> (meter)
  2:  Depth averaged U velocity <u velocity component> (meter per sec)
  3:  Depth averaged V velocity <v velocity component> (meter per sec)
Time: 2161 steps with dt=3600.0s
      2021-01-01 00:00:00 -- 2021-04-01 00:00:00

https://www.paraview.org/pipermail/paraview-developers/2015-October/004072.html

https://github.com/Chrismarsh/vtk-paraview-datetimefilter